# Retrieval-Augmented Generation (RAG)
## Information Retrieval and Search
### Keyword Search

In [140]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("gpreda/bbc-news")
news_data = pd.read_csv(os.path.join(path, "bbc_news.csv"))
news_data.head()

,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [141]:
import bm25s
corpus = list(news_data["title"] + " " + news_data["description"])
retriever = bm25s.BM25(corpus=corpus)

corpus_tokens = bm25s.tokenize(corpus)
print(f"Documents: {len(corpus_tokens.ids)}, Vocab: {len(corpus_tokens.vocab)}")
retriever.index(corpus_tokens)

query = "Retrieval augmented generation"
query_tokens = bm25s.tokenize(query)
docs, scores = retriever.retrieve(query_tokens, k=3)
print(f"Best result (score: {scores[0, 0]:.2f}): {docs[0, 0]}")

Documents: 42115, Vocab: 36405


Best result (score: 4.34): How brands hope to tempt you with videos on labels Companies are using augmented reality labels on their products to share videos and animations.


In [173]:
# {24538: 'augmented', 3129: 'generation'}
id_query = {corpus_tokens.vocab[q]: q for q in list(query_tokens.vocab) if q in corpus_tokens.vocab}

# [19281, 17907, 18662]
id_doc = [corpus.index(d) for d in docs[0]]

count = pd.DataFrame({d: {id_query[t]: corpus_tokens.ids[d].count(t) for t in id_query} for d in id_doc}).T
score = pd.Series(scores[0], index=id_doc, name="score")

In [172]:
count.join(news_data).join(score)

,augmented,generation,title,pubDate,guid,link,description,Score
19281,1,0,How brands hope to tempt you with videos on la...,"Sun, 16 Jul 2023 23:04:17 GMT",https://www.bbc.co.uk/news/business-66166552,https://www.bbc.co.uk/news/business-66166552?a...,Companies are using augmented reality labels o...,4.339485
17907,1,0,Vision Pro: Apple's new augmented reality head...,"Mon, 05 Jun 2023 21:18:23 GMT",https://www.bbc.co.uk/news/technology-65809408,https://www.bbc.co.uk/news/technology-65809408...,The high price and two-hour battery life raise...,3.871426
18662,0,2,Windrush generation recall arrival in Britain,"Thu, 22 Jun 2023 00:31:47 GMT",https://www.bbc.co.uk/news/uk-65963870,https://www.bbc.co.uk/news/uk-65963870?at_medi...,"The Windrush generation shaped modern Britain,...",3.823741


### Semantic Search

In [143]:
import sentence_transformers